# SeaHub image registry, metadata reconciliation, and embryo snips

This notebook treats the SeaHub image and metadata trees as read-only. All generated tables, QC previews, and crops are written beneath this notebook's directory.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from IPython.display import display

WORK_DIR = Path.cwd().resolve()
if not (WORK_DIR / "seahub_workflow.py").is_file():
    WORK_DIR = Path("results/nlammers/20260723_seahub").resolve()
sys.path.insert(0, str(WORK_DIR))

from seahub_workflow import (
    DEFAULT_IMAGE_ROOT,
    DEFAULT_METADATA_PATH,
    build_image_registry,
    export_registry_tables,
    load_collection_metadata,
    reconcile_with_collection_metadata,
    run_grounding_dino_segmentation,
    segmentation_candidates,
)

OUTPUT_DIR = WORK_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"Work directory: {WORK_DIR}")
print(f"Image input:    {DEFAULT_IMAGE_ROOT}")
print(f"Metadata input: {DEFAULT_METADATA_PATH}")

## What lives in the beta directory?

`results/mcolon/20260408_segmenting_sequence_images/01_test_gdino.py` is a CPU GroundingDINO detection test. It loads the fine-tuned checkpoint, predicts normalized `individual embryo` boxes for PNGs, converts `cxcywh` to `xyxy`, draws previews, and writes a detection CSV. It does not yet parse SeaHub, join collection metadata, require eight instances, assign within-FOV positions, or write individual crops. The functions used below add those pieces while keeping the same prompt, checkpoint, and initial thresholds.

## 1. Build the JPG registry

The registry includes abandoned/not-used paths and existing individual crops, but flags them so downstream segmentation can exclude them without losing provenance. Reading image dimensions takes roughly one minute for the current 1,925 files.

In [ ]:
REBUILD_REGISTRY = False
cached_registry_path = OUTPUT_DIR / "image_registry.csv"
if REBUILD_REGISTRY or not cached_registry_path.is_file():
    registry = build_image_registry(DEFAULT_IMAGE_ROOT)
else:
    registry = pd.read_csv(cached_registry_path, low_memory=False)
    print(f"Loaded cached registry: {cached_registry_path}")
print(f"Registered {len(registry):,} JPG/JPEG files")
display(registry.head())

In [ ]:
registry_summary = (
    registry.groupby(["image_role", "excluded_path"], dropna=False)
    .size()
    .rename("n_images")
    .reset_index()
)
display(registry_summary)

display(
    registry.groupby(["perturbation_domain", "image_role"])
    .size()
    .rename("n_images")
    .reset_index()
)

## 2. Read and reconcile collection metadata

`morphseq-env` does not contain `openpyxl`, so `load_collection_metadata` uses a small standard-library XLSX reader. Matching is conservative: experiment, collected stage, and normalized perturbation must agree; prior/addition stage is also used when encoded. Ambiguity remains visible in explicit status and candidate-count columns.

In [ ]:
collection_metadata = load_collection_metadata(DEFAULT_METADATA_PATH)
print(f"Loaded {len(collection_metadata):,} populated metadata rows")
display(collection_metadata.head())

In [ ]:
reconciled = reconcile_with_collection_metadata(registry, collection_metadata)
registry_csv, reconciliation_csv = export_registry_tables(
    registry, reconciled, OUTPUT_DIR
)
print(f"Wrote {registry_csv}")
print(f"Wrote {reconciliation_csv}")

In [ ]:
active_fovs = reconciled[
    reconciled["image_role"].eq("eight_embryo_fov")
    & ~reconciled["excluded_path"]
].copy()

display(
    active_fovs["metadata_match_status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="n_active_fovs")
)
display(
    active_fovs.groupby(["experiment_id", "metadata_match_status"])
    .size()
    .unstack(fill_value=0)
)

In [ ]:
candidates = segmentation_candidates(reconciled)
print(f"Unique high-confidence segmentation candidates: {len(candidates):,}")

preferred = candidates[
    candidates["experiment_id"].eq("GENE16")
    & candidates["filename"].eq("24hpf_A_foxc1a.jpg")
]
sample_row = (preferred if len(preferred) else candidates).iloc[0]
display(
    sample_row[[
        "image_id", "experiment_id", "filename", "fov_label",
        "perturbation_parsed", "stage_hpf", "stage_source_label",
        "metadata_collection_name", "metadata_match_status",
    ]].to_frame("value")
)

In [ ]:
with Image.open(sample_row["image_path"]) as sample_image:
    plt.figure(figsize=(12, 9))
    plt.imshow(sample_image)
    plt.title(
        f"{sample_row['experiment_id']} | {sample_row['filename']} | "
        f"{sample_row['metadata_collection_name']}"
    )
    plt.axis("off")
    plt.show()

## 3. GroundingDINO detection and eight embryo snips

Positions are assigned as top-row left-to-right (1–4), then bottom-row left-to-right (5–8). The filename's letter remains `fov_label`. The collected stage is numeric in `stage_hpf`; non-hour source labels such as `12s` remain available in `stage_source_label` without an invented conversion. Crops are emitted only when exactly eight detections survive NMS; mismatches get a QC preview and no misleading partial set. Every passing embryo is saved in both RGB and grayscale. These are padded detection-box snips, not pixelwise masks, because the beta model supplies boxes.

Model loading takes about 70 seconds on CPU and inference about 10 seconds per FOV in this environment. The cell is opt-in to avoid accidentally launching a full run.

In [ ]:
RUN_SEGMENTATION = False
SEGMENTATION_LIMIT = 1

if RUN_SEGMENTATION:
    # Start with the displayed sample. Replace this selection deliberately
    # before scaling to more rows.
    selected = pd.DataFrame([sample_row]).head(SEGMENTATION_LIMIT)
    embryo_manifest, segmentation_qc = run_grounding_dino_segmentation(
        selected,
        OUTPUT_DIR / "segmentation_notebook_run",
        device="cpu",
    )
    display(segmentation_qc)
    display(embryo_manifest.sort_values("embryo_position"))
else:
    print("Segmentation skipped. Set RUN_SEGMENTATION=True for the selected rows.")

## 4. Inspect the completed real smoke test

The checked-in exploratory output below was generated by the real fine-tuned checkpoint for the preferred GENE16 FOV at box/text thresholds 0.15/0.10.

In [ ]:
smoke_dir = OUTPUT_DIR / "segmentation_smoke_test"
smoke_manifest_path = smoke_dir / "embryo_manifest.csv"
smoke_qc_path = smoke_dir / "segmentation_qc.csv"
color_contact_sheet_path = smoke_dir / "snips_contact_sheet.jpg"
grayscale_contact_sheet_path = smoke_dir / "snips_grayscale_contact_sheet.jpg"

if smoke_qc_path.is_file():
    smoke_qc = pd.read_csv(smoke_qc_path)
    smoke_manifest = pd.read_csv(smoke_manifest_path, low_memory=False)
    display(smoke_qc)
    display(
        smoke_manifest[[
            "embryo_position", "stage_hpf", "detection_confidence",
            "snip_color_path", "snip_grayscale_path",
            "metadata_collection_name", "perturbation_parsed",
        ]].sort_values("embryo_position")
    )
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    with Image.open(color_contact_sheet_path) as color_sheet:
        axes[0].imshow(color_sheet)
    with Image.open(grayscale_contact_sheet_path) as grayscale_sheet:
        axes[1].imshow(grayscale_sheet, cmap="gray")
    axes[0].set_title("Color snips: positions 1–8")
    axes[1].set_title("Grayscale snips: positions 1–8")
    for axis in axes:
        axis.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Smoke-test outputs are not present.")

## Decisions and next obstacles

- Abandoned/not-used images remain registered but are excluded by default.
- Existing 256×576/576×256 single-embryo images remain registered but are not re-segmented.
- Only unique exact/high-confidence metadata matches enter the default candidate table; duplicate or contradictory metadata requires review.
- Filename abbreviations (`her1,7`), stage disagreements, and chemical filenames that omit temperature account for many unresolved rows.
- The configured sandbox GroundingDINO checkout is absent. The support module uses the existing read-only source checkout and exact fine-tuning config, with the pure-PyTorch operator fallback.
- A scalable next step is a small curated alias/override CSV keyed by `image_id`, followed by a batched run and manual review of every count mismatch.